# Useful Resources
JAX Beginner Guide: https://docs.jax.dev/en/latest/beginner_guide.html#beginner-guide

Deep Learning Tutorials: https://uvadlc-notebooks.readthedocs.io/en/latest/index.html

JAX GitHub: https://github.com/jax-ml/jax

JAX Libraries: https://github.com/n2cholas/awesome-jax

Flax: https://flax.readthedocs.io/en/latest/

XLA: https://openxla.org/xla

Autodiff in JAX: https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html

Visualizing Graphs: https://bnikolic.co.uk/blog/python/jax/2022/02/22/jax-outputgraph-rev.html

LAX: https://docs.jax.dev/en/latest/jax.lax.html

In [1]:
import os

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import sys

np.random.seed(42)

In [2]:
np.__version__,  mpl.__version__, sys.version

('1.26.4', '3.9.4', '3.11.11 (main, Dec 11 2024, 16:28:39) [GCC 11.2.0]')

# Which devices are available

If the GPU is available on your device JAX will automatically use that as its default device. CPU/GPU should be set before importing jax/jax related libraries

In [3]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
# os.environ["JAX_PLATFORMS"] = "cpu" 

import jax
import jax.numpy as jnp
jax.__version__, 

('0.4.30',)

In [4]:
# If a GPU is available this will print
#   ('gpu', [cuda(id=0)])

# If there is no GPU available or if you set `os.environ["JAX_PLATFORMS"] = "cpu"` this will print
#   ('cpu', [CpuDevice(id=0)])

jax.default_backend(), jax.devices()

('gpu', [cuda(id=0)])

# Change floating point precision

JAX defaults to `float32` (~7 digits, $\epsilon\approx10^{-7}$). `float64` gives ~16 digits, but costs memory and is slower on the GPU.

Keep `float32` for neural network training.

Use `float64` for algorithms involving linear solves, inversions, or determinants. The error in solving $A\mathbf{x}=\mathbf{b}$ grows like $\kappa(A)\epsilon$.

Set this flag before creating any arrays that should be in `float64`.

In [5]:
jax.config.update("jax_enable_x64", True)
jnp.array(1.0).dtype

dtype('float64')

Computing $\dfrac{1-\cos(x)}{x^2}$, at $x=0.0001$

The true results is $0.5$

In [6]:
for dt in (jnp.float32, jnp.float64):
    x = jnp.array(1e-4, dt)
    print(f"Precision {dt.__name__}: {(1.0 - jnp.cos(x)) / x**2}")

Precision float32: 0.0
Precision float64: 0.4999999969612645


Rewriting as $\dfrac{2\sin(x/2)}{x^2}$

In [7]:
for dt in (jnp.float32, jnp.float64):
    x = jnp.array(1e-4, dt)
    print(f"Precision {dt.__name__}: {2 * jnp.sin(x / 2)**2 / x**2}")

Precision float32: 0.5
Precision float64: 0.4999999995833334


In [8]:
jax.config.update("jax_enable_x64", False)
jnp.array(1.0).dtype

dtype('float32')

# NumPy vs. JAX

JAX arrays are backed by XLA (Accelerated Linear Algebra) which is a compiler that optimizes operations commonly used in data science involving tensors

In [9]:
ones_jnp = jnp.ones((3,2))
ones_np = np.ones(shape=(3,2))

type(ones_jnp), type(ones_np)

(jaxlib.xla_extension.ArrayImpl, numpy.ndarray)

# Operations between NumPy arrays and JAX arrays

Priority is given to jax

In [10]:
type(ones_jnp + ones_np)

jaxlib.xla_extension.ArrayImpl

In [11]:
type(np.array(ones_jnp) + ones_np)

numpy.ndarray

Mixing NumPy and JAX has a cost. Every JAX operation has a small fixed overhead (the time JAX spends checking inputs, looking up the compiled kernel, allocating the output and launching the work on the device, before any math happens). 

This takes a few microseconds on CPU, more on GPU, and NumPy doesn't pay it. With small arrays, that overhead takes up most of the runtime, so if one JAX array slips into a NumPy calculation it can run much slower. With large arrays, JAX is usually faster. Rule of thumb: use NumPy for small bookkeeping arrays and JAX for the heavy math.

In [12]:
N = 1_000

a_np = np.random.rand(N)
a_jnp = jnp.array(a_np)

b_np = np.random.rand(N)
b_jnp = jnp.array(b_np)

In [13]:
%%timeit
# Fastest
a_np + b_np

1.15 µs ± 4.96 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [14]:
%%timeit
# Slow
(a_jnp + b_jnp).block_until_ready() # use .block_until_ready() due to asynchronous dispatch

302 µs ± 18.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%%timeit
# Slowest
(a_np + b_jnp).block_until_ready()

426 µs ± 23.2 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [16]:
N = 10_000_000

a_np = np.random.rand(N)
a_jnp = jnp.array(a_np)

b_np = np.random.rand(N)
b_jnp = jnp.array(b_np)

In [17]:
%%timeit
# Slow
a_np + b_np

15.7 ms ± 17.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [19]:
%%timeit
# Fastest
(a_jnp + b_jnp).block_until_ready()

838 µs ± 20.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
%%timeit
# Slowest
(a_np + b_jnp).block_until_ready()

18.7 ms ± 452 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


# How to change devices

`jax.device_get`: GPU to CPU

In [22]:
ones_jnp.devices()

{cuda(id=0)}

In [23]:
ones_jnp_cpu = jax.device_get(ones_jnp)
type(ones_jnp_cpu)

numpy.ndarray

`jax.device_put`: CPU to GPU

In [24]:
ones_jnp_gpu = jax.device_put(ones_jnp_cpu)
ones_jnp.devices(), type(ones_jnp_gpu)

({cuda(id=0)}, jaxlib.xla_extension.ArrayImpl)

# Modifying JAX and NumPy arrays

In [27]:
np_arange = np.arange(0, 8, 2)
np_arange

array([0, 2, 4, 6])

In [28]:
np_arange[-1] = 0.1
np_arange

array([0, 2, 4, 0])

# What went wrong here?

In [29]:
np_arange.dtype, type(0.1)

(dtype('int64'), float)

In [32]:
np_arange = np.arange(0, 8, 2, dtype=np.float32) # you could also do np.arange(0.0, 8.0, 2)
np_arange

array([0., 2., 4., 6.], dtype=float32)

In [33]:
np_arange[-1] = 0.1
np_arange

array([0. , 2. , 4. , 0.1], dtype=float32)

In [35]:
jnp_arange = jnp.arange(0, 8, 2, dtype=jnp.float32)
jnp_arange

Array([0., 2., 4., 6.], dtype=float32)

In [36]:
jnp_arange[-1] = 0.1

TypeError: '<class 'jaxlib.xla_extension.ArrayImpl'>' object does not support item assignment. JAX arrays are immutable. Instead of ``x[idx] = y``, use ``x = x.at[idx].set(y)`` or another .at[] method: https://jax.readthedocs.io/en/latest/_autosummary/jax.numpy.ndarray.at.html

### JAX arrays are immutable

Unlike NumPy, you can't change a JAX array in place. This is deliberate: JAX is built
around *pure* functions, which only read their inputs and only produce their outputs,
with no side effects. That's what makes it safe for JAX to transform them with `jit`,
`vmap`, and `grad`.

So inside a function, don't modify arrays defined outside it. Pass in what you need
and return what you change.

To "update" an array, use `x.at[idx].set(value)`. It returns a *new* array with the
change applied. Under `jit`, XLA usually reuses the original memory, so the copy is free.


In [37]:
jnp_arange = jnp_arange.at[-1].set(0.1)
jnp_arange

Array([0. , 2. , 4. , 0.1], dtype=float32)

# Random Numbers

Purity applies to random number generation too. NumPy hides a global seed inside the module and mutates it on every call, so `np.random.normal()` returns something different each time you run it — the function depends on state you can't see and can't pass around.

JAX makes that state explicit: you hand every sampling call a *key*, and the same key always gives the same result. Reproducibility comes for free, and the RNG works under `jit` and `vmap` like any other function.


In [80]:
np_random_num1 = np.random.normal()
np_random_num2 = np.random.normal()
np_random_num1, np_random_num2

(-0.4577878641446004, -1.7357579432602035)

This is not ideal for reproducibility. We are calling the same function, which the same inputs (none) and getting completely different results.

In [85]:
key = jax.random.PRNGKey(42)
key

Array([ 0, 42], dtype=uint32)

In [83]:
jax_random_num1 = jax.random.normal(key)
jax_random_num2 = jax.random.normal(key)
jax_random_num1.item(), jax_random_num2.item()

(-0.1847117692232132, -0.1847117692232132)

Though each time we sample this distribution we wouldn't expect to get the same exact output. Remember, these functions should have no side-effects, the same input means the same output.

To generate new or "unique" samples we use a new key that is dependent on the previous key.

In [86]:
key, subkey1, subkey2 = jax.random.split(key, 3)
key, subkey1, subkey2

(Array([3134548294, 3733159049], dtype=uint32),
 Array([3746501087,  894150801], dtype=uint32),
 Array([ 801545058, 2363201431], dtype=uint32))

In [87]:
jax_random_num1 = jax.random.normal(subkey1)
jax_random_num2 = jax.random.normal(subkey2)

jax_random_num1.item(), jax_random_num2.item()

(0.10796153545379639, -1.2226542234420776)

# Derivatives

In [88]:
def polynomial(x):
    return x**2 + 4*x + 5

In [ ]:
def polynomial(coefficients, x):
    # All three are eequivalent
    return jnp.polyval(coefficients, x)

    # powers = jnp.arange(len(coefficients))
    # return jnp.sum(coefficients * x[...,None]**powers, axis=-1)

    # result = 0
    # for c in coefficients:
        # result = result * x + c
    # return result

In [ ]:
# x^2 + 4x + 5 at x = 1
x = jnp.array(1.0)
coefficients = jnp.array([1.0, 4.0, 5.0])
polynomial(coefficients, x).item()

10.0

In [114]:
dy_dx = jax.grad(polynomial, argnums=1)
dy_dx(coefficients, x).item()

6.0

If `coefficients` never changes then you can simplify a little

In [113]:
dy_dx = jax.grad(lambda _x: polynomial(coefficients, _x)) # jax.grad defaults to argnums=0
dy_dx(x).item()

6.0

To take derivatives with respect to other arguments

In [115]:
def function_2d(x, y):
    return x**2 + y**2

In [116]:
x = 2.0
y = 1.0

df_dx = jax.grad(function_2d, argnums=0)(x, y)
df_dy = jax.grad(function_2d, argnums=1)(x, y)

df_dx.item(), df_dy.item()

(4.0, 2.0)

If you want both at once

In [117]:
df_dx, df_dy = jax.grad(function_2d, argnums=(0,1))(x, y)
df_dx.item(), df_dy.item()

(4.0, 2.0)

If you need the derivatives and the function value

In [118]:
value, grad = jax.value_and_grad(function_2d, argnums=(0,1))(x, y)
df_dx, df_dy = grad
value.item(), df_dx.item(), df_dy.item()

(5.0, 4.0, 2.0)

If our function has more than one output (save you want to print some intermediate calculation)

In [119]:
def new_function(x, y):
    z = x + y
    return x**2 + y**2, z

In [120]:
(value, aux), grad = jax.value_and_grad(new_function, argnums=0, has_aux=True)(x, y)
value.item(), aux.item(), grad.item()

(5.0, 3.0, 4.0)

For higher order derivatives, stack `grad`

In [122]:
def example_function(x):
    return x**5

In [123]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)
x = jax.random.normal(subkey)

grad_fn = jax.grad

# d^n/dx^n (x^n) = n! (doesnt depend on x)
x.item(), grad_fn(grad_fn(grad_fn(grad_fn(grad_fn(example_function)))))(x).item() 

(-1.251538872718811, 120.0)

Alternate way for demonstration

In [ ]:
# d^n/dx^n (x^{n+j}) = 0, j>1
g_fn = grad_fn(example_function)
for i in range(4):
    g_fn = grad_fn(g_fn)
g_fn(x).item() 

0.0

A lot of the time in machine learning we end up taking derivatives of functions that output scalars but what if our output is not? This calls for the Jacobian.

In [127]:
def vector_function(vector):
    new_vector = 3.0 * vector
    result = new_vector + 2.0
    return result

In [128]:
vector = jnp.array([-1.0, 3.0])

In [129]:
jax.grad(vector_function)(vector)

TypeError: Gradient only defined for scalar-output functions. Output had shape: (2,).

In [130]:
jax.jacobian(vector_function)(vector)

Array([[3., 0.],
       [0., 3.]], dtype=float32)

The Jacobian is given by
$$
\nabla \mathbf{f} =
\begin{bmatrix}
    \dfrac{\partial \mathbf{f}}{\partial x_1} & \cdots & \dfrac{\partial \mathbf{f}}{\partial x_n}
\end{bmatrix} =
\begin{bmatrix}
    \nabla^T f_1 \\
    \vdots \\
    \nabla^T f_m
\end{bmatrix} =
\begin{bmatrix}
    \dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n} \\
    \vdots & \ddots & \vdots \\
    \dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n}
\end{bmatrix}
$$
In our case
$$ f(\mathbf{X}) = 3X + 2, \quad \mathbf{X}\in\mathbb{R}^{2}$$

$$ \mathbf{f} = [f_1, f_2]^T, \quad \mathbf{X} = [x_1, x_2]^T$$

The first element of $\mathbf{f}$ is not influenced by the second element of $\mathbf{X}$ and vice-versa so the anti-diagonal is zero  

$$
\nabla \mathbf{f} =
\begin{bmatrix}
    \dfrac{\partial f_1}{\partial x_1} & \dfrac{\partial f_1}{\partial x_2} \\
    \dfrac{\partial f_2}{\partial x_1} & \dfrac{\partial f_2}{\partial x_2}
\end{bmatrix} =
\begin{bmatrix}
    3 & 0 \\
    0 & 3
\end{bmatrix}
$$

# Something more complicated

The powerful part about this is that no matter how comlicated our function, as long as we have the function we can take its derivative

$$
\begin{gather*}f(x, y, z, w, v) = \\\dfrac{\sin\left(e^{x^2 + y^2} + \cosh(wz)\right) + \log\left(1 + w^2 \cosh(z) + v^3\right) - \dfrac{x^3 y^2}{z + e^{-w} + \tan(v)} + \arctan\left(\dfrac{y}{x + z^2 + \sinh(vw)}\right) e^{-\sin(w) + \cos(yz)} + \dfrac{\ln(1 + x^2 + y^2)}{1 + v^4 w^2}}{\dfrac{\text{erf}(xz)}{z^y}\left(1 + (5xy)^2 + y^2 + z^2 + w^2 + v^2 + e^{-|x+y+z|} + \dfrac{\cosh(w)}{\log_2\left(\frac{x}{y}\right)}\right)}
\end{gather*}
 $$

In [131]:
def crazy_function(x, y, z, w, v):
    numerator = (
        jnp.sin(jnp.exp(x**2 + y**2) + jnp.cosh(w * z)) +
        jnp.log(1.0 + w**2 * jnp.cosh(z) + v**3) -
        (x**3 * y**2) / (z + jnp.exp(-w) + jnp.tan(v)) +
        jnp.arctan(y / (x + z**2 + jnp.sinh(v * w))) * jnp.exp(-jnp.sin(w) + jnp.cos(y * z)) +
        jnp.log(1.0 + x**2 + y**2) / (1.0 + v**4 * w**2)
    )

    denominator = jax.lax.erf(x * z) / (z**y) * \
                  (1.0 + (5.0 * x * y)**2 + y**2 + z**2 + w**2 + v**2 + jnp.exp(-jnp.abs(x + y + z)) + (jnp.cosh(w) / (jnp.log2(x / y))))
    return numerator / denominator

In [137]:
# generate 5 positve inputs

key = jax.random.PRNGKey(1234)
vals = abs(jax.random.normal(key, shape=(5,)))
[val.item() for val in vals]

[0.9066547155380249,
 0.9453375935554504,
 0.08205656707286835,
 1.843636155128479,
 0.459509015083313]

In [ ]:
derivative = jax.grad(crazy_function, argnums=(0,1,2,3,4))(*vals) # unpack the 5 values each as an argument

[deriv.item() for deriv in derivative] # df/dx, ..., df/dv

[3.7483572959899902,
 -4.122462272644043,
 -0.0038567520678043365,
 0.1159186065196991,
 -0.006834930740296841]

In [139]:
%%timeit
jax.grad(crazy_function, argnums=(0,1,2,3,4))(*vals)[0].block_until_ready()

54.4 ms ± 498 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


What if we have several vectors we want to input and take the derivatives of

# JIT (Just-In-Time compilation)

JIT is a way of calling XLA to look at a function and optimize it

In [142]:
slow_function = jax.grad(crazy_function, argnums=(0,1,2,3,4))
fast_function = jax.jit(jax.grad(crazy_function, argnums=(0,1,2,3,4)))

In [ ]:
slow_function(*vals), fast_function(*vals); # warmup

In [143]:
%%timeit
slow_function(*vals)

53.6 ms ± 720 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [144]:
%%timeit
fast_function(*vals)

150 µs ± 39.5 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


Generally you want to JIT your outer most function, and JIT any (jax) function involving heavy computations

# vmap (Vectorizing map)
- Note some of these example might not actually *need* `vmap` but they are here for demonstration

Let's say we have a dataset with a shape of $(B,N,d)$ and a function we want to evalute each of these examples on.

This dataset contains $B$ examples, each containing $N$ points of dimension $d$

Example: You have a dataset of 50 4D functions evaluated at 1 points in some domain so the datset is simplified to $(B,d)$ and you want to find the Jacobain for each of the 50 functions. We expect the output to be $(50,4,4)$. Each example in the dataset is $d=4$ and the Jacobain will gives a $(4,4)$ matrix. 

In [152]:
def nonlinear_fn(x):
    return jnp.tanh(x) * jnp.sin(x) + jnp.cos(x**2)

In [155]:
B, d = 50, 4
key = jax.random.PRNGKey(1)
dataset = jax.random.normal(key, (B, d))

If we just do

In [159]:
jax.jacobian(nonlinear_fn)(dataset).shape

(50, 4, 50, 4)

`jax.jacobian` treats the whole (50, 4) batch as one big input, so it computes the derivative of every output entry with respect to every input entry (not what we want)

Instead we use `jax.vmap` and tell it that we want to "loop" or vectorize over the first axis of the input. 

Writing `jax.vmap(function, in_axes=0)(input)` applies function to each slice of input along its first axis and stacks the results. It gives the same answer as a for loop over the rows, but runs as a single vectorized operation

In [ ]:
batched_jacobian = jax.vmap(jax.jacobian(nonlinear_fn), in_axes=0) # len(in_axes) = len(function inputs)
batched_jacobian(dataset).shape

(50, 4, 4)

In [162]:
%%timeit
batched_jacobian(dataset).block_until_ready()

7.31 ms ± 15.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


This is equivalent to (but much slower)

In [ ]:
jac_fn = jax.jacobian(nonlinear_fn)
jnp.stack([jac_fn(row) for row in dataset]).block_until_ready(); # warmup

In [167]:
%%timeit
jacobians = []
for row in dataset:
    jacobians.append(jac_fn(row))
jnp.stack(jacobians).block_until_ready()

288 ms ± 2.98 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


When using vmap, you can specify `in_axes` to be multiple things such as vectorzing over one axis while keeping another fixed

For a function with two inputs, `in_axes=(a, b)` tells `vmap` how to treat each of the function's two inputs. `a` applies to the first input and `b` to the second. An integer means split that input along that axis, so each call gets one slice. `None` means pass that input whole to every call. The outputs of all the calls are stacked into one result.

We want to compare every point in `X` with every point in `Y` using `kernel`, which only takes **two single points** and returns a scalar. `X` has `M = 6` points and `Y` has `N = 4` points, so the result should be a `(6, 4)` matrix.

In [ ]:
def kernel(x1, x2): # two single points -> scalar
    return jnp.exp(-0.5 * jnp.sum((x1 - x2)**2))

key = jax.random.PRNGKey(0)
subkey1, subkey2 = jax.random.split(key)

M, N = 6, 4
X = jax.random.normal(subkey1, (M,2)) # M points in 2D
Y = jax.random.normal(subkey2, (N,2)) # N points in 2D

`in_axes` has one entry per argument of `kernel`. `None` means "use this argument as-is on every call" and `0` means "go through this argument one row at a time".

So `in_axes=(None, 0)` keeps `x1 = X[0]` fixed and steps through the rows of `Y`, giving one value per point in `Y`, which is one row of the matrix with shape `(4,)`.

In [ ]:
# Hold x1 (first point in X) fixed, sweep over all of Y -> one row of length N
kernel_row = jax.vmap(kernel, in_axes=(None,0))
kernel_row(X[0], Y).shape

(4,)

Now we `vmap` the row function. `in_axes=(0, None)` steps through the rows of `X` and passes **all** of `Y` in on every call. Each call produces one row, and `vmap` stacks them, giving shape `(6, 4)`.

The `None` matters here. With `in_axes=0`, `vmap` would step through `X` and `Y` *together* (`X[0]` with `Y[0]`, `X[1]` with `Y[1]`, ...). That gives only matched pairs rather than every combination, and it fails here since `X` and `Y` have different lengths.

In [185]:
# Now sweep x1 over all of X -> (M, N)
gram = jax.vmap(kernel_row, in_axes=(0,None))
gram(X, Y).shape # (6, 4)

(6, 4)

In [ ]:
# in the above code we also just used a stack vmap
# where we could just written jax.vmap(jax.vmap(kernel, in_axes=(None,0)), in_axes=(0,None))(X, Y)

In [186]:
%%timeit
gram = jax.vmap(kernel_row, in_axes=(0,None))
gram(X, Y).block_until_ready()

1.86 ms ± 7.36 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The above cell is equivalant to (but of course slower)

In [189]:
K_loop = np.zeros((6,4))
for i in range(6):
    for j in range(4):
        K_loop[i,j] = kernel(X[i], Y[j])

In [192]:
np.allclose(K_loop, gram(X, Y))

True

### `vmap` summary and best practices

Use `vmap` whenever you would otherwise write a for loop over examples. Write the function for a **single** input, and let `vmap` handle the batch.

`jax.vmap(fun, in_axes=0, out_axes=0)`

- `fun`: the function to vectorize, written for one example
- `in_axes`: one entry per argument, giving the axis to loop over. Use `None` for arguments that should be passed in whole on every call (shared parameters, a fixed point, a constant)
- `out_axes`: where the batch axis goes in the output (the default `0` puts it first)

**Best practices**
- **Get the single-example version working first.** Call `fun` on one input and check the output shape before adding `vmap`.
- **Check against a loop.** On a small input, compare the `vmap` result with a plain for loop using `jnp.allclose`. This is the quickest way to catch a wrong `in_axes`.
- **Think "which arguments change from call to call?"** Those get an axis (usually `0`); everything else gets `None`.
- **Nest `vmap` for every combination.** Each `vmap` adds one axis: the inner `vmap` becomes the last axis of the output and the outer one the first.
- **Combine with `jit` and `grad`.** `vmap` works with the other transformations, e.g. `jax.jit(jax.vmap(f))` or `jax.vmap(jax.grad(f))` for per-example gradients.
- **Not every loop can be vectorized.** If each step depends on the previous one (time stepping, recurrences), use `jax.lax.scan` instead.

**Don't worry if this doesn't click right away.** Most people don't find `vmap` intuitive at first. Getting comfortable with it, especially `in_axes` and nesting, takes practice and some trial and error. When it gets confusing, write the for loop first, then turn it into a `vmap` and check that the shapes match.


# Bonus

### More on stacking `vmap`

Stacking `vmap` lets us loop over more than one thing. Here we build a multiplication table: every value in `x` times every value in `y`. `multiply` only takes two single numbers.

In [194]:
def multiply(x, y):
    return x * y

x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([10.0, 20.0])

The first `vmap` keeps one `x` fixed and steps through `y`. This gives one row of the table.

In [ ]:
row = jax.vmap(multiply, in_axes=(None,0))
row(x[0], y)    # [10. 20.]

Array([10., 20.], dtype=float32)

The second `vmap` steps through `x` and passes all of `y` to `row` each time. Each call gives one row, and `vmap` stacks them into a `(3, 2)` table.

In [ ]:
table = jax.vmap(row, in_axes=(0,None))
table(x, y)

Array([[10., 20.],
       [20., 40.],
       [30., 60.]], dtype=float32)

: 